# Stage D - BaCP internals, forward over teachers

**Question: does each additional teacher earn its place?**

C3 -> **D1** (+SnC, snapshot teachers) -> **D2** (+PrC, pretrained teacher) = **full BaCP**.

D2 *is* BaCP. Everything above it is the path there, and every rung below it is a
competing explanation that has been ruled out or not.

### Teacher order was corrected before any run
The order is **FiC -> SnC -> PrC**, not the pre-registered SnC -> PrC -> FiC. The
original would have made C2 -> C3 change the teacher **and** the objective form in
one step - the same two-changes-one-label error the submitted paper's Table 6 makes.

A forward ladder measures each component *conditional on the prefix already
installed*, so the order is part of the claim. It is fixed in the manifest in
advance, and Stage D4 is what detects order-sensitivity.

**Gate G5**: if both +SnC and +PrC have CIs containing zero, stop before tuning -
do not tune a method whose own components are indistinguishable.

> Paired, forward over teachers.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))          # so ladder_nb is importable
import ladder_nb as nb
info = nb.setup()


## Configure

`TIER` 1 is the spine (5 seeds). Cells per GPU and dataloader workers are constants at the top of `pool.py`.


In [ ]:
TIER    = 1
GPUS    = info['gpus'] or 1
SEEDS   = None          # None = every seed the tier schedules

RUNGS = ['D1', 'D2']

import manifest as M
grid = [c for c in M.cells(TIER, rungs=RUNGS)]
print(f'{len(grid)} cell(s) planned over {len(set(c["rung"] for c in grid))} rung(s)')
for r in RUNGS:
    n = sum(1 for c in grid if c['rung'] == r)
    print(f'  {r:10s} {n} seed(s)' if n else f'  {r:10s} -- NOT IN TIER {TIER}')


## Run

Idempotent - a cell is complete iff a record carrying its key exists, so re-running skips what is done. Dense cells run first as a hard barrier. Safe to interrupt; you lose at most the cells in flight.


In [ ]:
summary = nb.run_stage(RUNGS, tier=TIER, gpus=GPUS, seeds=SEEDS)
print(summary['ok'], 'ok,', summary['failed'], 'failed,', summary['skipped'], 'skipped')


## Progress and health

The `nan` column is the one to read first. A diverged run sits at exactly 10.0969% (chance on CIFAR-10) for the rest of training and every delta computed from it is meaningless.


In [ ]:
nb.progress(TIER)


## Watch a single cell

Use this when something looks wrong - it streams one line per epoch so you can see *where* a run breaks rather than only that it did.


In [ ]:
cell = nb.attach(nb.pick('D1', seed=1, tier=TIER))
nb.show(cell)
# hist, first_nan = nb.watch(cell, gpu=0)
# nb.plot(hist, first_nan, cell['key'])


## Table and gates


In [ ]:
out = nb.report(TIER)
